#Volume & Completeness Test (Count Check)

In [0]:
# Notebook: 06_bronze_quality_tests (FIXED)
from pyspark.sql.functions import count

CATALOG = "vstone_catalog"
BRONZE = "bronze"
CHUNKS_PATH = f"/Volumes/{CATALOG}/raw/chunks"
LANDING_PATH = f"/Volumes/{CATALOG}/raw/landing"

# Updated mapping with correct CSV options for multi-line text
reconciliation_map = {
    "listings_csv_copyinto": (f"{CHUNKS_PATH}/1_main_chunk_1.csv", "csv", {"header": "true"}),
    "listings_csv_dlt": (f"{CHUNKS_PATH}/1_main_chunk_2.csv", "csv", {"header": "true"}),
    "listings_json_autoloader": (f"{CHUNKS_PATH}/1_main_chunk_3.json", "json", {"multiLine": "true"}),
    "listings_xml_pyspark": (f"{CHUNKS_PATH}/1_main_chunk_4.xml", "xml", {"rowTag": "record"}),
    
    # FIXED: Added multiLine and escape for 1_text.csv
    "listings_text_bronze": (f"{LANDING_PATH}/1_text.csv", "csv", {"header": "true", "multiLine": "true", "escape": '"'}),
    
    "listings_photo_bronze": (f"{LANDING_PATH}/1_photo.csv", "csv", {"header": "true"}),
    "car_catalog_bronze": (f"{LANDING_PATH}/catalogs.csv", "csv", {"header": "true", "sep": ";"}),
    "geo_locations_bronze": (f"{LANDING_PATH}/final_geografic.csv", "csv", {"header": "true"})
}

print("📊 RAW-TO-BRONZE RECONCILIATION TESTING (WITH MULTILINE FIX):")
print("-" * 80)

for table, (src_path, fmt, opts) in reconciliation_map.items():
    # 1. Get Source Count (Now correctly reading multi-line CSVs)
    raw_count = spark.read.format(fmt).options(**opts).load(src_path).count()
    
    # 2. Get Bronze Table Count
    bronze_count = spark.table(f"{CATALOG}.{BRONZE}.{table}").count()
    
    # 3. Compare
    diff = raw_count - bronze_count
    status = "✅ PASSED" if diff == 0 else f"❌ FAILED ({diff} mismatch)"
    
    print(f"{status} | Table: {table:<25} | Raw: {raw_count:,} | Bronze: {bronze_count:,}")

print("-" * 80)

# Row to row integrity test

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# 1. CONFIGURATION (Matching your source to bronze mapping)
test_config = [
    {"name": "Chunk 1 (CSV)", "src": "/Volumes/vstone_catalog/raw/chunks/1_main_chunk_1.csv", "table": "listings_csv_copyinto", "fmt": "csv", "opts": {"header": "true"}},
    {"name": "Chunk 2 (CSV/DLT)", "src": "/Volumes/vstone_catalog/raw/chunks/1_main_chunk_2.csv", "table": "listings_csv_dlt", "fmt": "csv", "opts": {"header": "true"}},
    {"name": "Chunk 3 (JSON)", "src": "/Volumes/vstone_catalog/raw/chunks/1_main_chunk_3.json", "table": "listings_json_autoloader", "fmt": "json", "opts": {"multiLine": "true"}},
    {"name": "Chunk 4 (XML)", "src": "/Volumes/vstone_catalog/raw/chunks/1_main_chunk_4.xml", "table": "listings_xml_pyspark", "fmt": "xml", "opts": {"rowTag": "record"}},
    {
        "name": "Text Data", 
        "src": "/Volumes/vstone_catalog/raw/landing/1_text.csv", 
        "table": "listings_text_bronze", 
        "fmt": "csv", 
        "opts": {
            "header": "true", 
            "multiLine": "true", # <--- CRITICAL FIX
            "escape": '"'        # <--- CRITICAL FIX
        }
    },
    {"name": "Photo Data", "src": "/Volumes/vstone_catalog/raw/landing/1_photo.csv", "table": "listings_photo_bronze", "fmt": "csv", "opts": {"header": "true"}},
    {"name": "Catalog Master", "src": "/Volumes/vstone_catalog/raw/landing/catalogs.csv", "table": "car_catalog_bronze", "fmt": "csv", "opts": {"header": "true", "sep": ";"}},
    {"name": "Geo Data", "src": "/Volumes/vstone_catalog/raw/landing/final_geografic.csv", "table": "geo_locations_bronze", "fmt": "csv", "opts": {"header": "true"}}
]

# 2. CORE TEST FUNCTION (Clean String Comparison)
def run_detailed_integrity_test(source_df, bronze_df, table_alias):
    # Sirf wahi columns jo dono taraf hain (Audit columns load_dt, source_file ignore ho jayenge)
    common_cols = [c for c in source_df.columns if c in bronze_df.columns]
    
    def generate_fingerprints(df, columns):
        # NORMALIZATION: 
        # 1. Sabko String mein cast karein
        # 2. Trim spaces (White spaces verify karne ke liye)
        # 3. Handle NULLs (Jo COPY INTO mein aa sakte hain) as empty strings
        
        temp_df = df.select([
            F.coalesce(F.trim(F.col(c).cast("string")), F.lit("")).alias(c) 
            for c in columns
        ])
        
        # SHA-256 Fingerprint generation
        return (
            temp_df.withColumn("fingerprint", F.sha2(F.concat_ws("||", *columns), 256))
                   .select("fingerprint")
        )

    source_fp = generate_fingerprints(source_df, common_cols)
    bronze_fp = generate_fingerprints(bronze_df, common_cols)

    # Reconciliation using subtract
    missing_in_bronze = source_fp.subtract(bronze_fp).count()
    extra_in_bronze = bronze_fp.subtract(source_fp).count()
    mismatch_total = missing_in_bronze + extra_in_bronze

    if mismatch_total == 0:
        print(f"✅ {table_alias}: Row-to-Row Data Integrity 100% Verified.")
    else:
        print(f"❌ {table_alias}: INTEGRITY FAILURE! Mismatches: {mismatch_total}")
        print(f"   -> Missing in Bronze: {missing_in_bronze} | Extra in Bronze: {extra_in_bronze}")

# 3. EXECUTE LOOP
print("🕵️ STARTING FINAL STRING-ONLY INTEGRITY TESTS...")
print("-" * 75)

for cfg in test_config:
    try:
        # Step A: Read Source as String (No inferSchema)
        source_df = (spark.read.format(cfg["fmt"])
                     .options(**cfg["opts"])
                     .option("inferSchema", "false") 
                     .load(cfg["src"]))
        
        # Step B: Read Bronze Table
        bronze_df = spark.table(f"vstone_catalog.bronze.{cfg['table']}")
        
        # Step C: Run Test
        run_detailed_integrity_test(source_df, bronze_df, cfg["name"])
        
    except Exception as e:
        print(f"⚠️ Error testing {cfg['name']}: {str(e)}")

print("-" * 75)

# Detailed report

In [0]:
from pyspark.sql.functions import lit
import pandas as pd

# 1. Reconciliation Logic
results = []

def perform_full_reconciliation(name, src_path, table_name, fmt, opts):
    try:
        # A. Source Data (Raw)
        src_df = spark.read.format(fmt).options(**opts).option("inferSchema", "false").load(src_path)
        src_count = src_df.count()
        
        # B. Bronze Data (Target)
        brz_df = spark.table(f"vstone_catalog.bronze.{table_name}")
        brz_count = brz_df.count()
        
        # C. Detailed Integrity Check (Fingerprinting)
        common_cols = [c for c in src_df.columns if c in brz_df.columns]
        
        def get_fp(df, cols):
            return df.select([F.coalesce(F.trim(F.col(c).cast("string")), F.lit("")).alias(c) for c in cols]) \
                     .withColumn("fp", F.sha2(F.concat_ws("||", *cols), 256)).select("fp")
        
        src_fp = get_fp(src_df, common_cols)
        brz_fp = get_fp(brz_df, common_cols)
        
        mismatches = src_fp.subtract(brz_fp).count() + brz_fp.subtract(src_fp).count()
        
        # Store results
        status = "✅ PASSED" if (src_count == brz_count and mismatches == 0) else "❌ FAILED"
        results.append({
            "File Name": name,
            "Source Count": src_count,
            "Bronze Count": brz_count,
            "Count Diff": src_count - brz_count,
            "Integrity Mismatches": mismatches,
            "Status": status
        })
    except Exception as e:
        results.append({"File Name": name, "Status": f"⚠️ ERROR: {str(e)}"})

# 2. Run for all 8 Tables
for cfg in test_config:
    perform_full_reconciliation(cfg["name"], cfg["src"], cfg["table"], cfg["fmt"], cfg["opts"])

# 3. Display Detailed Summary Report
summary_df = spark.createDataFrame(pd.DataFrame(results))
display(summary_df)

#Schema & Meta-Character Test

In [0]:
import pyspark.sql.functions as F

def run_detailed_metadata_test(table_fullname):
    print(f"\n📊 TABLE: {table_fullname}")
    df = spark.table(table_fullname)
    cols = df.columns
    
    # --- 1. SCHEMA VALIDATION (Russian Character Support) ---
    if "car_catalog_bronze" in table_fullname:
        russian_cols = [c for c in cols if any(ord(char) > 127 for char in c)]
        if len(russian_cols) > 0:
            print(f"  ✅ SCHEMA PASSED: Russian headers detected ({len(russian_cols)} columns).")
        else:
            print("  ❌ SCHEMA FAILED: No Russian headers found. Check Column Mapping!")
    
    # --- 2. AUDIT METADATA TEST (load_dt & source_file) ---
    expected_audit = ["load_dt", "source_file"]
    missing_audit = [c for c in expected_audit if c not in cols]
    
    if not missing_audit:
        null_audit = df.filter("load_dt IS NULL OR source_file IS NULL").count()
        if null_audit == 0:
            print("  ✅ METADATA PASSED: Audit columns (load_dt, source_file) are 100% populated.")
        else:
            print(f"  ❌ METADATA FAILED: Found {null_audit} rows with missing audit info!")
    else:
        print(f"  ❌ METADATA FAILED: Missing Audit Columns: {missing_audit}")

    # --- 3. UPDATED RESCUED DATA TEST (Should be 0 now) ---
    if "_rescued_data" in cols:
        rescued_count = df.filter("_rescued_data IS NOT NULL").count()
        if rescued_count == 0:
            print("  ✅ SCHEMA INTEGRITY: 100% Clean. No data loss in '_rescued_data'.")
        else:
            # Display sample of remaining issues if any
            print(f"  ⚠️ ALERT: {rescued_count} records still have corrupted fields!")
            df.filter("_rescued_data IS NOT NULL").select("_rescued_data").limit(1).show(truncate=False)

# --- EXECUTION LOOP FOR ALL BRONZE TABLES ---
all_tables = [
    "vstone_catalog.bronze.listings_csv_copyinto",
    "vstone_catalog.bronze.listings_csv_dlt",
    "vstone_catalog.bronze.listings_json_autoloader",
    "vstone_catalog.bronze.listings_xml_pyspark",
    "vstone_catalog.bronze.listings_text_bronze",
    "vstone_catalog.bronze.listings_photo_bronze",
    "vstone_catalog.bronze.car_catalog_bronze",
    "vstone_catalog.bronze.geo_locations_bronze"
]

print("🕵️ STARTING FINAL BRONZE METADATA & SCHEMA AUDIT...")
print("-" * 75)
for table in all_tables:
    try:
        run_detailed_metadata_test(table)
    except Exception as e:
        print(f"⚠️ Error accessing {table}: {str(e)}")
print("-" * 75)